# 01 — Limpeza e Anonimização

**Entrada**: `data/raw/survey_responses.xlsx` (32 respostas, 63 colunas, Google Forms)

**Saídas**:
- `data/processed/anonymized.csv` — base limpa, sem PII, schema estável
- `data/processed/likert_importance.csv` — 13 características × 32 respondentes em forma long
- `data/processed/likert_priority.csv` — idem para prioridade
- `data/processed/skills.csv` — 10 atividades × 32 respondentes
- `data/processed/open_responses.csv` — todas respostas abertas em forma long
- `data/codebook/codebook_pt.md` — schema documentado

**Critérios de qualidade**:
- Confirmar 32 linhas × 63 colunas
- Validar TCLE em todas as respostas
- Drop email + coluna `@dropdown` artefato
- Normalizar estado → UF de 2 letras
- Mapear todos os Likerts para escala ordinal 1–N
- Detectar menções nominais nas abertas (regex pra emails e palavras com inicial maiúscula no meio de frase)

In [1]:
import re
import sys
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, str(Path.cwd()))
import utils as U

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 200)

## 1. Carregamento + validação de shape

In [2]:
df = U.load_raw()
assert df.shape == (32, 63), df.shape
print(f"Shape: {df.shape}")
df.head(3)

Shape: (32, 63)


,timestamp,consent,age,state,gender,education,role,seniority,n_projects,skill_cleaning,skill_normalization,skill_outliers,skill_integration,skill_transformation,skill_validation,skill_pipelines,skill_monitoring,skill_libs,skill_split,word_1,word_2,word_3,word_4,word_5,re_experience,imp_precision,imp_completeness,imp_consistency,imp_credibility,imp_currentness,imp_accessibility,imp_compliance,imp_reliability,imp_efficiency,imp_traceability,imp_understandability,imp_availability,imp_recoverability,imp_justification,pri_precision,pri_completeness,pri_consistency,pri_credibility,pri_currentness,pri_accessibility,pri_compliance,pri_reliability,pri_efficiency,pri_traceability,pri_understandability,pri_availability,pri_recoverability,pri_justification,balance_open,versioning_open,incorporation_open,measurement_open,discussion_freq,documentation_open,challenges_open,support_freq,_email_drop,_dropdown_drop
0,2025-02-14 15:15:25.144,Aceito participar,18-24 anos,Ceará,Homem,Ensino superior,Cientista de dados,Júnior (até 5 anos),2,Média,Média,Média,Média,Média,Média,Média,Média,Média,Média,Precisão,Consistência,Completude,Relevância,Viés,"Sim, foi uma ótima experiência, embora complic...",Importante,Importante,Importante,Importante,Importante,Importante,Importante,Importante,Importante,Importante,Importante,Importante,Importante,NaN,Alta prioridade,Alta prioridade,Alta prioridade,Alta prioridade,Alta prioridade,Alta prioridade,Alta prioridade,Alta prioridade,Alta prioridade,Alta prioridade,Alta prioridade,Alta prioridade,Alta prioridade,NaN,"Equilibrar precisão, compreensibilidade e efic...",Garante a consistência e rastreabilidade dos d...,Avaliação inicial durante a coleta e preparaçã...,Análise de métricas de performance (ex.: preci...,"Menos de uma vez por semana, mas pelo menos um...","Linguagem estruturada (texto), Ferramentas de ...",Inconsistência entre diferentes fontes de dado...,Ocasionalmente,NaN,NaN
1,2025-02-14 15:46:31.820,Aceito participar,25-34 anos,PE,Mulher,Estudante de Doutorado,Cientista de dados,Júnior (até 5 anos),2,Muito alto,Muito alto,Muito alto,Média,Acima da média,Acima da média,Média,Média,Acima da média,Muito alto,Fonte,Estabilidade,Necessário,Essencial,Determinante,Não,Muito importante,Importante,Muito importante,Muito importante,Muito importante,Muito importante,Muito importante,Muito importante,Muito importante,Importante,Muito importante,Muito importante,Muito importante,NaN,Essencial,Alta prioridade,Essencial,Essencial,Essencial,Essencial,Essencial,Essencial,Essencial,Alta prioridade,Essencial,Essencial,Alta prioridade,NaN,"A prioridade é na precisão dos dados, seguido ...",Garante a consistência e rastreabilidade dos d...,Avaliação inicial durante a coleta e preparaçã...,Análise de métricas de performance (ex.: preci...,"Pelo menos uma vez por semana, mas não todos o...","Linguagem estruturada (texto), Reuniões de Ali...","Dados incompletos ou ausentes, Dados desatuali...",Ocasionalmente,NaN,NaN
2,2025-02-14 15:53:04.364,Aceito participar,25-34 anos,Ceará,Homem,Mestrado,Cientista de dados,Pleno (6 a 9 anos),13,Muito alto,Muito alto,Muito alto,Acima da média,Muito alto,Muito alto,Média,Média,Acima da média,Muito alto,Confiabilidade,Disponibilidade,Atualização,Integridade,Processamento,"Geralmente, são os primeiros passos quando co...",Muito importante,Muito importante,Muito importante,Muito importante,Muito importante,Muito importante,Muito importante,Muito importante,Muito importante,Muito importante,Muito importante,Muito importante,Muito importante,NaN,Alta prioridade,Alta prioridade,Alta prioridade,Essencial,Essencial,Essencial,Essencial,Essencial,Neutro,Alta prioridade,Alta prioridade,Alta prioridade,Alta prioridade,NaN,Tudo vai depender de como nossa aplicação prec...,Garante a consistência e rastreabilidade dos d...,Avaliação inicial durante a coleta e preparaçã...,Análise de métricas de performance (ex.: preci...,"Menos de uma vez por semana, mas pelo menos um...","Linguagem estruturada (texto), 

## 2. Validação TCLE

Todos os 32 respondentes precisam ter aceito o TCLE. Drop quem não aceitou.

In [3]:
consent_counts = df["consent"].value_counts(dropna=False)
print(consent_counts)
assert (df["consent"] == "Aceito participar").all(), "Existe respondente que não aceitou TCLE!"
print("\n[OK] Todas as 32 respostas aceitaram o TCLE.")

consent
Aceito participar    32
Name: count, dtype: int64

[OK] Todas as 32 respostas aceitaram o TCLE.


## 3. Detecção de duplicados (timestamp + perfil)

In [4]:
n_dup_timestamp = df["timestamp"].duplicated().sum()
n_dup_perfil = df.duplicated(subset=["age", "state", "gender", "role", "seniority", "n_projects"]).sum()
print(f"Duplicatas por timestamp: {n_dup_timestamp}")
print(f"Duplicatas por perfil demográfico: {n_dup_perfil}")
print(f"Janela de coleta: {df['timestamp'].min()} → {df['timestamp'].max()}")

Duplicatas por timestamp: 0
Duplicatas por perfil demográfico: 1
Janela de coleta: 2025-02-14 15:15:25.144000 → 2025-03-15 16:34:41.757000


## 4. Drop colunas com PII / vazias

- `_email_drop` — coluna de e-mail (Q23) — anonimização
- `_dropdown_drop` — coluna `@dropdown` artefato do Google Forms (sempre NaN)

In [5]:
n_emails = df["_email_drop"].notna().sum()
print(f"E-mails informados (serão removidos): {n_emails}/32")
assert df["_dropdown_drop"].isna().all(), "Coluna @dropdown deveria ser totalmente vazia."
df = df.drop(columns=["_email_drop", "_dropdown_drop", "consent"])
print(f"Shape após drops: {df.shape}")

E-mails informados (serão removidos): 15/32
Shape após drops: (32, 60)


## 5. Normalização de estado → UF

Respostas livres têm variantes ("Ceará", "CE", "PARAÍBA", etc.). Padroniza para sigla 2 letras.

In [6]:
df["state_raw"] = df["state"]
df["state"] = df["state_raw"].apply(U.normalize_state)
df["region"] = df["state"].map(U.UF_TO_REGION)

print("Mapeamento estado:")
print(df[["state_raw", "state", "region"]].value_counts(dropna=False).to_string())

missing_state = df[df["state"].isna()]
if len(missing_state):
    print("\n[ALERTA] Estados não mapeados:", missing_state["state_raw"].tolist())
else:
    print("\n[OK] Todos os estados mapeados para UF.")

Mapeamento estado:
state_raw           state  region  
Ceará               CE     Nordeste    9
Ceará               CE     Nordeste    6
CE                  CE     Nordeste    2
São Paulo           SP     Sudeste     2
Alagoas             AL     Nordeste    2
RJ                  RJ     Sudeste     2
PE                  PE     Nordeste    1
PR                  PR     Sul         1
Rio Grande do Sul   RS     Sul         1
BA                  BA     Nordeste    1
Bahia               BA     Nordeste    1
PARAÍBA             PB     Nordeste    1
Rio de Janeiro      RJ     Sudeste     1
Amazonas            AM     Norte       1
Rio de Janeiro      RJ     Sudeste     1

[OK] Todos os estados mapeados para UF.


## 6. Mapeamento de Likerts para escala ordinal

- Habilidade Q8: 1 (muito baixa) → 5 (muito alta), "Não se aplica" → NaN
- Importância Q11: 1 (nada importante) → 5 (muito importante)
- Prioridade Q13: 1 (sem prioridade) → 5 (essencial)
- Discussão Q19: 1 (nunca) → 5 (todos os dias)
- Suporte Q22: 1 (raramente) → 4 (sempre)

Manter colunas originais como `*_raw` para auditoria.

In [7]:
def apply_map(df: pd.DataFrame, cols: list[str], mapping: dict, suffix: str = "_raw") -> pd.DataFrame:
    """Renomeia col→col_raw e cria nova col com mapping aplicado. Falha se valor não estiver em mapping."""
    for c in cols:
        df[c + suffix] = df[c]
        unknown = set(df[c].dropna().unique()) - set(mapping.keys())
        if unknown:
            raise ValueError(f"Valores inesperados em {c}: {unknown}")
        df[c] = df[c].map(mapping).astype("Int64")
    return df

df = apply_map(df, U.SKILL_COLS, U.SKILL_MAP)
df = apply_map(df, U.IMP_COLS, U.IMPORTANCE_MAP)
df = apply_map(df, U.PRI_COLS, U.PRIORITY_MAP)
df = apply_map(df, ["discussion_freq"], U.DISCUSSION_FREQ_MAP)
df = apply_map(df, ["support_freq"], U.SUPPORT_FREQ_MAP)

df["seniority_ordinal"] = df["seniority"].map(U.SENIORITY_ORDINAL).astype("Int64")
df["seniority_group"] = df["seniority"].map(U.SENIORITY_GROUP)
df["role_group"] = df["role"].map(U.ROLE_GROUP)

print("Likerts mapeados. Sanity check (importância precisão):")
print(df["imp_precision"].value_counts(dropna=False).sort_index())
print("\nGrupo senioridade:")
print(df["seniority_group"].value_counts(dropna=False))

Likerts mapeados. Sanity check (importância precisão):
imp_precision
1     1
4     4
5    27
Name: count, dtype: Int64

Grupo senioridade:
seniority_group
senior    20
junior    12
Name: count, dtype: int64


## 7. Inspeção de respostas abertas

Quantas respostas não-vazias por questão e procura por menções nominais (suspeita de identificação).

In [8]:
OPEN_COLS = [
    "re_experience", "imp_justification", "pri_justification",
    "balance_open", "versioning_open", "incorporation_open",
    "measurement_open", "documentation_open", "challenges_open",
]
summary = pd.DataFrame({
    "question": OPEN_COLS,
    "n_responses": [df[c].notna().sum() for c in OPEN_COLS],
    "avg_chars": [int(df[c].dropna().str.len().mean()) if df[c].notna().any() else 0 for c in OPEN_COLS],
    "max_chars": [int(df[c].dropna().str.len().max()) if df[c].notna().any() else 0 for c in OPEN_COLS],
})
print(summary.to_string(index=False))

          question  n_responses  avg_chars  max_chars
     re_experience           32        175       1009
 imp_justification           12        278        642
 pri_justification            9        228        623
      balance_open           31        318       1733
   versioning_open           32        136        138
incorporation_open           32        160        253
  measurement_open           32         89        543
documentation_open           32        104        220
   challenges_open           32        166        344


In [9]:
# Detecção de menções nominais. Heurística simples: e-mails e tokens com Inicial Maiúscula
# que não são início de frase. Falsos positivos são tolerados — saída é só pra revisão manual.
EMAIL_RE = re.compile(r"[\w.+-]+@[\w-]+\.[\w.-]+")
PROPER_NAME_RE = re.compile(r"(?<![\.!?]\s)(?<!^)\b([A-ZÁÉÍÓÚÂÊÔÃÕÇ][a-záéíóúâêôãõç]{2,})\b")

STOP_PROPER = {
    # Itens que aparecem com inicial maiúscula mas não são nomes próprios
    "Brasil", "Python", "Java", "JavaScript", "SQL", "Pandas", "PySpark", "Dask",
    "PCA", "Excel", "Power", "Bi", "Microsoft", "Google", "AWS", "GCP", "Azure",
    "Linux", "Mac", "Windows", "Github", "Git", "Gitlab", "Bitbucket", "Jira",
    "Confluence", "Slack", "Teams", "Zoom", "Notion", "Tableau",
    "Apache", "Spark", "Hadoop", "Kafka", "Airflow", "Mlflow", "TensorFlow",
    "PyTorch", "Sklearn", "Scikit", "Numpy", "Scipy", "Matplotlib",
    "Tcle", "Ml", "Ia", "Br", "Ex", "Etl", "Api", "Crud", "Mvp", "Poc",
    "Engenharia", "Requisitos", "Projeto", "Equipe", "Cliente", "Aprendizado",
    "Maquina", "Máquina", "Dados", "Qualidade", "Modelo", "Modelos",
}

alerts = []
for col in OPEN_COLS:
    for idx, txt in df[col].dropna().items():
        emails = EMAIL_RE.findall(txt)
        proper = [m for m in PROPER_NAME_RE.findall(txt) if m not in STOP_PROPER]
        if emails or proper:
            alerts.append({
                "row": idx,
                "col": col,
                "emails": emails,
                "proper_name_candidates": proper[:5],
                "snippet": txt[:200],
            })

alerts_df = pd.DataFrame(alerts)
print(f"Alertas pra revisão manual: {len(alerts_df)}")
if len(alerts_df):
    print(alerts_df.to_string())

Alertas pra revisão manual: 111
     row                 col emails                                               proper_name_candidates                                                                                                                                                                                                    snippet
0      4       re_experience     []                                                            [Ciência]                                                                                                                                Já participei de todas as fases do ciclo de um projeto em Ciência de Dados.
1     29       re_experience     []                   [Histórias, Cenários, Funcionais, Não, Funcionais]   Participei na melhoria de requisitos do meu projeto atual. Nesse processo sempre é feita uma reunião com o cliente onde ele faz o levantamento dos cenários e casos que ele deseja obter. Ao final dessa
2     29   imp_justification     []         

## 8. Inspeção das 5 palavras (Q9)

Texto curtíssimo, normalizar lowercase + strip. Reportar contagens brutas (lemmatização vai pro notebook 04).

In [10]:
for c in U.WORD_COLS:
    df[c + "_raw"] = df[c]
    df[c] = df[c].astype("string").str.strip().str.lower().replace({"": pd.NA})

word_long = df[U.WORD_COLS].melt(value_name="word", var_name="position").dropna()
word_long["position"] = word_long["position"].str.replace("word_", "").astype(int)
print(f"Total tokens: {len(word_long)} (esperado até 32×5=160)")
print("Top-15 brutos:")
print(word_long["word"].value_counts().head(15))

Total tokens: 160 (esperado até 32×5=160)
Top-15 brutos:
word
consistência       10
precisão            7
relevância          7
completude          7
confiabilidade      6
atualização         6
qualidade           4
quantidade          4
acurácia            3
balanceamento       2
corretude           2
limpeza             2
disponibilidade     2
diversidade         2
validade            2
Name: count, dtype: Int64


## 9. Relatório de qualidade (missing por coluna, n efetivo por questão)

In [11]:
missing = df.isna().sum().sort_values(ascending=False)
missing = missing[missing > 0]
print("Colunas com missing:")
print(missing.to_string())

Colunas com missing:
pri_justification       23
imp_justification       20
skill_monitoring         3
skill_cleaning           2
skill_outliers           2
skill_pipelines          2
skill_integration        2
skill_normalization      2
skill_split              1
skill_transformation     1
skill_libs               1
skill_validation         1
support_freq_raw         1
support_freq             1
balance_open             1


In [12]:
# Sanity check cruzado: idade × experiência × papel
print("Idade × Senioridade:")
print(pd.crosstab(df["age"], df["seniority"]))
print("\nPapel × N projetos (média):")
print(df.groupby("role")["n_projects"].agg(["count", "mean", "median"]).round(1))

Idade × Senioridade:


seniority   Estagiário  Júnior (até 5 anos)  Pleno (6 a 9 anos)  Sênior (10+ anos)
age                                                                               
18-24 anos           2                    3                   0                  0
25-34 anos           0                    6                   7                  4
35-44 anos           0                    1                   2                  4
45-54 anos           0                    0                   1                  2

Papel × N projetos (média):
                                                    count  mean  median
role                                                                   
Cientista de dados                                     18   9.1     5.0
Desenvolvedor de Software (Backend, front-end, ...      9   4.2     3.0
Engenheiro de Machine Learning                          1   3.0     3.0
Engenheiro de dados                                     1   5.0     5.0
Gerente de Dados e IA                    

## 10. Persistência

Salva 5 CSVs separados pra notebooks downstream consumirem sem reprocessar.

In [13]:
out = U.DATA_PROC / "anonymized.csv"
df.to_csv(out, index=False)
print(f"[saved] {out} — {df.shape}")

[saved] /home/emanuel/Projects/trabalho-marcelinho/data/processed/anonymized.csv — (32, 108)


In [14]:
# Long forms para análises agregadas
imp_long = df[U.IMP_COLS].assign(respondent_id=df.index).melt(
    id_vars="respondent_id", var_name="characteristic", value_name="importance"
).dropna()
imp_long["characteristic"] = imp_long["characteristic"].str.replace("imp_", "")
imp_long.to_csv(U.DATA_PROC / "likert_importance.csv", index=False)
print(f"[saved] likert_importance.csv — {imp_long.shape}")

pri_long = df[U.PRI_COLS].assign(respondent_id=df.index).melt(
    id_vars="respondent_id", var_name="characteristic", value_name="priority"
).dropna()
pri_long["characteristic"] = pri_long["characteristic"].str.replace("pri_", "")
pri_long.to_csv(U.DATA_PROC / "likert_priority.csv", index=False)
print(f"[saved] likert_priority.csv — {pri_long.shape}")

skills_long = df[U.SKILL_COLS].assign(respondent_id=df.index).melt(
    id_vars="respondent_id", var_name="activity", value_name="skill"
)
skills_long.to_csv(U.DATA_PROC / "skills.csv", index=False)
print(f"[saved] skills.csv — {skills_long.shape}")

open_long = df[OPEN_COLS].assign(respondent_id=df.index).melt(
    id_vars="respondent_id", var_name="question", value_name="response"
).dropna()
open_long.to_csv(U.DATA_PROC / "open_responses.csv", index=False)
print(f"[saved] open_responses.csv — {open_long.shape}")

word_long.to_csv(U.DATA_PROC / "words.csv", index=False)
print(f"[saved] words.csv — {word_long.shape}")

[saved] likert_importance.csv — (416, 3)
[saved] likert_priority.csv — (416, 3)


[saved] skills.csv — (320, 3)
[saved] open_responses.csv — (244, 3)


[saved] words.csv — (160, 2)


## 11. Codebook (PT) — schema documentado

Geração automática a partir do mapeamento em `utils.py`.

In [15]:
lines = ["# Codebook — Survey Qualidade de Dados em ML (PT)", ""]
lines.append(f"**N**: 32 respondentes · **Coleta**: {df['timestamp'].min().date()} → {df['timestamp'].max().date()}")
lines.append("")
lines.append("## Schema")
lines.append("")
lines.append("| Coluna | Tipo | Domínio | Origem (Q) |")
lines.append("|---|---|---|---|")
DOMAIN = {
    "timestamp": ("datetime", "timestamp Forms", "-"),
    "age": ("categórica", "18-24, 25-34, 35-44, 45-54", "Q1"),
    "state": ("UF (2 letras)", "AC..TO", "Q2"),
    "region": ("categórica", "Norte/Nordeste/Centro-Oeste/Sudeste/Sul", "derivada"),
    "gender": ("categórica", "Homem, Mulher", "Q3"),
    "education": ("categórica", "6 níveis", "Q4"),
    "role": ("categórica", "7 valores livres normalizados", "Q5"),
    "seniority": ("ordinal", "Estagiário<Júnior<Pleno<Sênior", "Q6"),
    "seniority_ordinal": ("int 1-4", "-", "derivada"),
    "seniority_group": ("categórica", "junior, senior", "derivada"),
    "n_projects": ("int", "0..40", "Q7"),
    "role_group": ("categórica", "data_scientist/developer/...", "derivada"),
}
for col, (t, dom, q) in DOMAIN.items():
    lines.append(f"| `{col}` | {t} | {dom} | {q} |")
for c in U.SKILL_COLS:
    lines.append(f"| `{c}` | Likert 1-5 | 1=Muito baixa..5=Muito alta | Q8 |")
for c in U.WORD_COLS:
    lines.append(f"| `{c}` | string | palavra livre lowercase | Q9 |")
for c in U.IMP_COLS:
    lines.append(f"| `{c}` | Likert 1-5 | 1=Nada importante..5=Muito importante | Q11 |")
for c in U.PRI_COLS:
    lines.append(f"| `{c}` | Likert 1-5 | 1=Sem prioridade..5=Essencial | Q13 |")
for c in ("re_experience", "imp_justification", "pri_justification",
          "balance_open", "versioning_open", "incorporation_open",
          "measurement_open", "documentation_open", "challenges_open"):
    lines.append(f"| `{c}` | string (open) | resposta livre | varia |")
lines.append("| `discussion_freq` | Likert 1-5 | 1=Nunca..5=Todos os dias | Q19 |")
lines.append("| `support_freq` | Likert 1-4 | 1=Raramente..4=Sempre | Q22 |")

lines.append("")
lines.append("## Anonimização")
lines.append("")
lines.append("- Coluna `email` (Q23): removida. 15 respondentes informaram e-mail. Não publicada.")
lines.append("- Coluna `@dropdown`: artefato Google Forms (sempre vazia). Removida.")
lines.append("- Estado: padronizado para UF de 2 letras + região derivada — não identifica indivíduo.")
lines.append("- Abertas: revisão por regex pra emails e nomes próprios (ver alertas no notebook).")

(U.DATA_CODEBOOK / "codebook_pt.md").write_text("\n".join(lines))
print(f"[saved] codebook_pt.md — {len(lines)} linhas")

[saved] codebook_pt.md — 79 linhas


## 12. Sumário final

In [16]:
print(f"N final: {len(df)}")
print(f"Colunas finais: {df.shape[1]}")
print(f"Colunas com algum missing: {(df.isna().sum() > 0).sum()}")
print(f"Tokens Q9 não-vazios: {len(word_long)}")
print(f"Respostas abertas não-vazias: {len(open_long)}")

N final: 32
Colunas finais: 108
Colunas com algum missing: 15
Tokens Q9 não-vazios: 160
Respostas abertas não-vazias: 244
